# Sales Forecasting with a 30-Day Moving Average

**Goal:** Forecast next month's sales using a 30-day moving-average baseline, visualize historical sales and forecast values, calculate MAE when actual values for the forecast period are available, and export forecast results to CSV.

> **Dataset requirement:** Use a CSV containing a date column and a numeric sales column. Set `DATE_COLUMN` and `SALES_COLUMN` to match your dataset. If no dataset is available, sample data is generated so the notebook can run end-to-end.

## 1. Import libraries and configure
Centralize dataset settings and forecast parameters here.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error

# Configuration
DATA_PATH = Path("sales.csv")       # Place your dataset beside this notebook
DATE_COLUMN = "date"                # Date column in your CSV
SALES_COLUMN = "sales"              # Numeric sales column
ROLLING_WINDOW = 30                 # Days used for moving average
USE_SAMPLE_DATA_IF_MISSING = True    # False to require your CSV
OUTPUT_PATH = Path("sales_forecast.csv")

print("Configuration loaded successfully.")

## 2. Load the sales data
If `sales.csv` exists, it is loaded. Otherwise sample daily data is generated (if enabled). Replace sample data with your real data before final submission.

In [ ]:
def create_sample_sales(start="2024-01-01", periods=900, seed=42):
    """Create reproducible sample daily sales with trend, weekly seasonality, and noise."""
    rng = np.random.default_rng(seed)
    dates = pd.date_range(start=start, periods=periods, freq="D")
    trend = np.linspace(100, 180, periods)
    weekly_pattern = 12 * np.sin(2 * np.pi * np.arange(periods) / 7)
    noise = rng.normal(0, 8, periods)
    sales_values = np.maximum(trend + weekly_pattern + noise, 0)
    return pd.DataFrame({DATE_COLUMN: dates, SALES_COLUMN: sales_values})

if DATA_PATH.exists():
    raw_df = pd.read_csv(DATA_PATH)
    print(f"Loaded dataset: {DATA_PATH.resolve()}")
elif USE_SAMPLE_DATA_IF_MISSING:
    raw_df = create_sample_sales()
    print("sales.csv not found; using reproducible sample data.")
else:
    raise FileNotFoundError(f"Dataset not found: {DATA_PATH.resolve()}")

raw_df.head()

## 3. Clean data and create a daily time series
Dates and sales are parsed, invalid rows dropped, duplicate dates aggregated, and missing calendar dates filled with zero. This assumes a missing date means no sales; change this assumption if missing data means unreported sales.

In [ ]:
def prepare_daily_sales(data, date_column, sales_column):
    """Validate and aggregate raw records into a sorted daily sales series."""
    required = {date_column, sales_column}
    missing = required.difference(data.columns)
    if missing:
        raise ValueError(f"Missing required columns: {sorted(missing)}")

    cleaned = data[[date_column, sales_column]].copy()
    cleaned[date_column] = pd.to_datetime(cleaned[date_column], errors="coerce")
    cleaned[sales_column] = pd.to_numeric(cleaned[sales_column], errors="coerce")
    cleaned = cleaned.dropna(subset=[date_column, sales_column])
    if cleaned.empty:
        raise ValueError("No valid rows remain after parsing the date and sales columns.")

    cleaned[date_column] = cleaned[date_column].dt.normalize()
    daily_series = (cleaned.groupby(date_column)[sales_column].sum()
                   .sort_index().asfreq("D", fill_value=0))
    daily_series.name = "sales"
    if len(daily_series) <= ROLLING_WINDOW:
        raise ValueError(f"At least {ROLLING_WINDOW + 1} daily observations are required.")
    return daily_series

sales = prepare_daily_sales(raw_df, DATE_COLUMN, SALES_COLUMN)
print(f"Daily observations: {len(sales):,}")
print(f"Date range: {sales.index.min().date()} to {sales.index.max().date()}")
sales.head()

## 4. Calculate the 30-day moving-average forecast
The forecast equals the mean of the latest 30 daily sales observations and is held constant for every day in the upcoming calendar month.

In [ ]:
rolling_mean = sales.rolling(window=ROLLING_WINDOW, min_periods=ROLLING_WINDOW).mean()
forecast_value = float(rolling_mean.iloc[-1])
last_date = sales.index.max()
forecast_start = (last_date + pd.offsets.MonthBegin(1)).normalize()
forecast_end = forecast_start + pd.offsets.MonthEnd(0)
forecast_dates = pd.date_range(forecast_start, forecast_end, freq="D")
forecast = pd.Series(forecast_value, index=forecast_dates, name="forecast_sales")

print(f"Forecast period: {forecast.index.min().date()} to {forecast.index.max().date()}")
print(f"Forecast per day: {forecast_value:,.2f}")
forecast.head()

## 5. Visualize historical sales and forecast
Plot daily historical sales, the 30-day rolling mean, and the forecast period together.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(sales.index, sales.values, label="Historical daily sales", linewidth=1.2)
ax.plot(rolling_mean.index, rolling_mean.values, label=f"{ROLLING_WINDOW}-day rolling mean", linewidth=2)
ax.plot(forecast.index, forecast.values, label="Next-month forecast", linewidth=2, linestyle="--")
ax.axvline(forecast.index[0], linestyle=":", label="Forecast start")
ax.set(title="Sales Forecast Using 30-Day Moving Average", xlabel="Date", ylabel="Sales")
ax.legend()
ax.grid(alpha=0.25)
fig.tight_layout()
plt.show()

## 6. Calculate MAE for the forecast period if actuals exist
MAE is only meaningful when actual values for the same forecast dates are available. This usually will not be possible for a future month; a historical backtest is provided next.

In [ ]:
actual_next_month = sales.reindex(forecast.index)
if actual_next_month.notna().all():
    mae_future = mean_absolute_error(actual_next_month.values, forecast.values)
    print(f"MAE for forecast period: {mae_future:,.2f}")
else:
    print("Actual sales for forecast dates are not available yet; see historical backtest below.")

## 7. Backtest on the last available calendar month
Reserve the latest month in the data for evaluation. Each day's prediction is calculated from the prior 30 days, so actual previous days are used as they would be in a rolling daily forecast. This produces an MAE from historical data.

In [ ]:
test_month_start = sales.index.max().to_period("M").start_time
test_actual = sales.loc[sales.index >= test_month_start]
predictions = []
for test_date in test_actual.index:
    prior_days = sales.loc[sales.index < test_date].tail(ROLLING_WINDOW)
    predictions.append(float(prior_days.mean()) if len(prior_days) == ROLLING_WINDOW else np.nan)

backtest = pd.DataFrame({"actual_sales": test_actual, "predicted_sales": predictions}, index=test_actual.index).dropna()
if not backtest.empty:
    backtest_mae = mean_absolute_error(backtest["actual_sales"], backtest["predicted_sales"])
    print(f"Backtest period: {backtest.index.min().date()} to {backtest.index.max().date()}")
    print(f"Backtest MAE: {backtest_mae:,.2f}")
    display(backtest.head())
else:
    print("Insufficient history to backtest the last month.")

## 8. Export forecast to CSV
Export forecast dates, values, window size, and model name to a CSV file in the current working directory.

In [ ]:
forecast_output = forecast.rename_axis("date").reset_index()
forecast_output["rolling_window_days"] = ROLLING_WINDOW
forecast_output["model"] = "30-day moving average"
forecast_output.to_csv(OUTPUT_PATH, index=False)
print(f"Saved forecast file: {OUTPUT_PATH.resolve()}")
forecast_output.head()

## 9. Project conclusion
- **Model:** 30-day moving-average baseline
- **Forecast horizon:** Upcoming calendar month
- **Visualization:** Historical sales, rolling average, forecast
- **Evaluation:** MAE on the last historical month (backtest)
- **Export:** `sales_forecast.csv`

If sample data was used, rerun this notebook with the actual sales CSV before submission. This baseline does not explicitly model holidays, promotions, or long-term seasonal patterns.